## Step 1: Install Required Packages

First, we need to install CrewAI and its dependencies. We'll use:
- `crewai`: The main framework (includes the `@tool` decorator we use for custom tools)
- `python-dotenv`: Loads credentials from a `.env` file

**Note:** CrewAI currently requires Python 3.10–3.13 and pins the MCP SDK to 1.x, which conflicts with the LiteLLM version used in the Google ADK notebook. Run this notebook in its own environment/kernel.

In [1]:
# Install required packages
%pip install -q "crewai>=1.15" python-dotenv

# Windows only: pywin32 (used by the MCP SDK) normally needs a kernel restart after installation.
# Re-reading the site-packages .pth files makes it importable right away.
import site, sys, sysconfig
if sys.platform == "win32":
    for _site_dir in (sysconfig.get_paths()["purelib"], site.getusersitepackages()):
        site.addsitedir(_site_dir)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\Kr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Step 2: Setup API Keys and Environment

To use Azure OpenAI, you need your resource's API key, endpoint, and model deployment name (Azure portal or Azure AI Foundry → your resource → **Keys and Endpoint**).

**Security Best Practice:** Store your API key in a `.env` file, not in the code! The cell below loads it from `.env` (in this folder or any parent folder) and prompts for anything missing.
```
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/openai/v1/
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [2]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


## Step 3: Create Custom Tools

Tools are functions that agents can use to perform specific tasks. Let's create custom tools for our movie production assistant.

### Tool 1: Script Sentiment Analyzer
Analyzes the emotional tone of a script excerpt.

In [3]:
from crewai.tools import tool
import re

@tool("Script Sentiment Analyzer")
def analyze_script_sentiment(script_text: str) -> str:
    """
    Analyzes the sentiment and emotional tone of a movie script excerpt.
    Returns sentiment score and dominant emotions.

    Args:
        script_text: A string containing the script excerpt to analyze
    """
    # Simple sentiment analysis based on keywords
    positive_words = ['love', 'happy', 'joy', 'wonderful', 'amazing', 'brilliant', 'triumph', 'success']
    negative_words = ['hate', 'sad', 'angry', 'terrible', 'awful', 'tragedy', 'failure', 'death']
    suspense_words = ['mystery', 'unknown', 'hidden', 'secret', 'suspense', 'thriller']

    script_lower = script_text.lower()

    pos_count = sum(1 for word in positive_words if word in script_lower)
    neg_count = sum(1 for word in negative_words if word in script_lower)
    sus_count = sum(1 for word in suspense_words if word in script_lower)

    total = pos_count + neg_count + sus_count

    if total == 0:
        return "Sentiment: Neutral - No strong emotional indicators detected."

    result = f"Sentiment Analysis Results:\n"
    result += f"- Positive tone: {pos_count}/{total} ({pos_count/total*100:.1f}%)\n"
    result += f"- Negative tone: {neg_count}/{total} ({neg_count/total*100:.1f}%)\n"
    result += f"- Suspenseful tone: {sus_count}/{total} ({sus_count/total*100:.1f}%)\n"

    if pos_count > neg_count and pos_count > sus_count:
        result += "\nOverall: Uplifting and positive story"
    elif neg_count > pos_count and neg_count > sus_count:
        result += "\nOverall: Dark and dramatic narrative"
    else:
        result += "\nOverall: Tense and mysterious atmosphere"

    return result

# Test the tool
test_script = "A tale of love and triumph as Sarah discovers the hidden mystery that brings her joy."
print(analyze_script_sentiment.run(test_script))

Sentiment Analysis Results:
- Positive tone: 3/5 (60.0%)
- Negative tone: 0/5 (0.0%)
- Suspenseful tone: 2/5 (40.0%)

Overall: Uplifting and positive story


### Tool 2: Budget Calculator
Calculates production costs based on different categories.

In [4]:
@tool("Production Budget Calculator")
def calculate_production_budget(category: str, days: int = 30, crew_size: int = 50) -> str:
    """
    Calculates estimated production budget for different movie categories.

    Args:
        category: Type of production (indie, medium, blockbuster)
        days: Number of shooting days
        crew_size: Number of crew members
    """
    # Base daily rates per category
    rates = {
        'indie': {'daily': 50000, 'crew': 500, 'equipment': 10000},
        'medium': {'daily': 200000, 'crew': 2000, 'equipment': 50000},
        'blockbuster': {'daily': 1000000, 'crew': 10000, 'equipment': 250000}
    }

    category = category.lower()
    if category not in rates:
        return f"Unknown category. Choose from: {', '.join(rates.keys())}"

    rate = rates[category]

    production_cost = rate['daily'] * days
    crew_cost = rate['crew'] * crew_size * days
    equipment_cost = rate['equipment'] * days
    contingency = (production_cost + crew_cost + equipment_cost) * 0.15

    total = production_cost + crew_cost + equipment_cost + contingency

    result = f"📊 Budget Breakdown for {category.upper()} Production:\n"
    result += f"\n🎬 Production Costs: ${production_cost:,}"
    result += f"\n👥 Crew Costs ({crew_size} members): ${crew_cost:,}"
    result += f"\n📹 Equipment Rental: ${equipment_cost:,}"
    result += f"\n🛡️ Contingency (15%): ${contingency:,}"
    result += f"\n\n💰 TOTAL ESTIMATED BUDGET: ${total:,}"
    result += f"\n📅 For {days} shooting days"

    return result

# Test the tool
print(calculate_production_budget.run("medium", days=45, crew_size=75))

📊 Budget Breakdown for MEDIUM Production:

🎬 Production Costs: $9,000,000
👥 Crew Costs (75 members): $6,750,000
📹 Equipment Rental: $2,250,000
🛡️ Contingency (15%): $2,700,000.0

💰 TOTAL ESTIMATED BUDGET: $20,700,000.0
📅 For 45 shooting days


### Tool 3: Soundtrack Genre Recommender
Suggests music genres based on the movie's theme and mood.

In [5]:
@tool("Soundtrack Genre Recommender")
def recommend_soundtrack_genre(movie_genre: str, mood: str) -> str:
    """
    Recommends soundtrack genres based on movie genre and mood.

    Args:
        movie_genre: The genre of the movie (action, drama, comedy, horror, sci-fi, romance)
        mood: The desired mood (intense, light, emotional, mysterious, upbeat)
    """
    recommendations = {
        'action': {
            'intense': ['Epic Orchestral', 'Electronic/Synth', 'Heavy Metal'],
            'light': ['Pop Rock', 'Funk', 'Electronic Pop'],
            'emotional': ['Cinematic Strings', 'Piano & Strings', 'Alternative Rock'],
        },
        'drama': {
            'intense': ['Classical', 'Dramatic Orchestral', 'Jazz Noir'],
            'emotional': ['Piano Solo', 'String Quartet', 'Acoustic Guitar'],
            'mysterious': ['Ambient', 'Minimalist Piano', 'Contemporary Classical'],
        },
        'comedy': {
            'upbeat': ['Jazz', 'Pop', 'Ska', 'Funk'],
            'light': ['Acoustic Pop', 'Ukulele', 'Whistling & Bells'],
        },
        'horror': {
            'intense': ['Dark Ambient', 'Industrial', 'Atonal Orchestral'],
            'mysterious': ['Theremin', 'Dissonant Strings', 'Electronic Drone'],
        },
        'sci-fi': {
            'intense': ['Electronic/Synth', 'Cyberpunk', 'Orchestral Hybrid'],
            'mysterious': ['Ambient Electronic', 'Experimental', 'Synth Pad'],
        },
        'romance': {
            'emotional': ['Classical Piano', 'Acoustic', 'Indie Folk'],
            'light': ['Soft Pop', 'Bossa Nova', 'Acoustic Guitar'],
            'upbeat': ['Pop', 'Soul', 'Light Jazz'],
        }
    }

    genre = movie_genre.lower()
    mood_key = mood.lower()

    if genre not in recommendations:
        return f"Unknown genre. Try: {', '.join(recommendations.keys())}"

    genre_moods = recommendations[genre]
    if mood_key not in genre_moods:
        return f"For {genre}, available moods are: {', '.join(genre_moods.keys())}"

    suggested = genre_moods[mood_key]

    result = f"🎵 Soundtrack Recommendations for {genre.upper()} ({mood} mood):\n\n"
    for i, genre_rec in enumerate(suggested, 1):
        result += f"{i}. {genre_rec}\n"

    result += f"\n💡 Tip: Consider mixing {suggested[0]} with {suggested[-1]} for dynamic scenes!"

    return result

# Test the tool
print(recommend_soundtrack_genre.run("sci-fi", "mysterious"))

🎵 Soundtrack Recommendations for SCI-FI (mysterious mood):

1. Ambient Electronic
2. Experimental
3. Synth Pad

💡 Tip: Consider mixing Ambient Electronic with Synth Pad for dynamic scenes!


## Step 4: Integrate MCP Tool (Model Context Protocol)

MCP tools allow agents to interact with external systems and APIs. Here we'll create a simulated MCP tool for accessing a movie database.

**Note:** In production, MCP tools connect to real external services. This is a simplified demonstration.

In [6]:
from typing import Dict, Any

@tool("Movie Market Research MCP")
def fetch_market_data(genre: str) -> str:
    """
    Fetches market research data for a specific movie genre from the database.
    This simulates an MCP tool that connects to external market research APIs.

    Args:
        genre: The movie genre to research (action, drama, comedy, horror, sci-fi, romance)
    """
    # Simulated market data (in production, this would call a real API)
    market_database = {
        'action': {
            'avg_box_office': '$450M',
            'audience_demographic': '18-35 years, 65% male',
            'peak_season': 'Summer (May-August)',
            'trending_themes': ['Superheroes', 'International espionage', 'Disaster scenarios'],
            'streaming_performance': 'High - 85% completion rate',
        },
        'drama': {
            'avg_box_office': '$120M',
            'audience_demographic': '30-60 years, 55% female',
            'peak_season': 'Awards season (October-February)',
            'trending_themes': ['Social justice', 'Biography', 'Family dynamics'],
            'streaming_performance': 'Medium - 72% completion rate',
        },
        'comedy': {
            'avg_box_office': '$180M',
            'audience_demographic': '18-45 years, balanced gender',
            'peak_season': 'Year-round, slight spike in holidays',
            'trending_themes': ['Romantic comedy', 'Workplace humor', 'Cultural clash'],
            'streaming_performance': 'Very High - 90% completion rate',
        },
        'horror': {
            'avg_box_office': '$95M',
            'audience_demographic': '16-30 years, 52% male',
            'peak_season': 'Fall (September-November)',
            'trending_themes': ['Psychological horror', 'Folk horror', 'Home invasion'],
            'streaming_performance': 'High - 82% completion rate',
        },
        'sci-fi': {
            'avg_box_office': '$380M',
            'audience_demographic': '18-40 years, 60% male',
            'peak_season': 'Summer and Holiday season',
            'trending_themes': ['AI & technology', 'Space exploration', 'Time travel'],
            'streaming_performance': 'High - 88% completion rate',
        },
        'romance': {
            'avg_box_office': '$85M',
            'audience_demographic': '18-50 years, 70% female',
            'peak_season': 'Valentine\'s Day season (January-February)',
            'trending_themes': ['Second chance romance', 'Holiday romance', 'Cross-cultural love'],
            'streaming_performance': 'Very High - 92% completion rate',
        },
    }

    genre_key = genre.lower()

    if genre_key not in market_database:
        return f"❌ No market data available for '{genre}'. Available genres: {', '.join(market_database.keys())}"

    data = market_database[genre_key]

    result = f"📈 Market Research Report: {genre.upper()} Genre\n"
    result += f"\n💰 Average Box Office: {data['avg_box_office']}"
    result += f"\n👥 Target Demographic: {data['audience_demographic']}"
    result += f"\n📅 Best Release Window: {data['peak_season']}"
    result += f"\n🎬 Trending Themes:"
    for theme in data['trending_themes']:
        result += f"\n   • {theme}"
    result += f"\n📺 Streaming Performance: {data['streaming_performance']}"
    result += f"\n\n✅ Data retrieved from Market Intelligence Database (MCP)"

    return result

# Test the MCP tool
print(fetch_market_data.run("action"))

📈 Market Research Report: ACTION Genre

💰 Average Box Office: $450M
👥 Target Demographic: 18-35 years, 65% male
📅 Best Release Window: Summer (May-August)
🎬 Trending Themes:
   • Superheroes
   • International espionage
   • Disaster scenarios
📺 Streaming Performance: High - 85% completion rate

✅ Data retrieved from Market Intelligence Database (MCP)


## Step 5: Configure the Language Model

Now let's set up our Azure OpenAI deployment to power our agents. Azure OpenAI's v1 API is OpenAI-compatible, so we use CrewAI's native OpenAI provider (`openai/<deployment>`) with `base_url` pointing at the Azure endpoint.

**Note:** GPT-5-family deployments are reasoning models that only support the default `temperature`, so we don't set it.

In [7]:
from crewai import LLM

# Initialize the Azure OpenAI deployment using CrewAI's LLM wrapper
llm = LLM(
    model=f"openai/{AZURE_OPENAI_DEPLOYMENT}",
    base_url=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

print("✅ Language model configured: Azure OpenAI")
print(f"   - Deployment: {AZURE_OPENAI_DEPLOYMENT}")

✅ Language model configured: Azure OpenAI
   - Deployment: gpt-5.4-mini


## Step 6: Create Specialized Agents

Agents are AI assistants with specific roles and tools. Let's create a crew of specialized agents for movie production:

1. **Script Analyst** - Analyzes scripts and provides creative feedback
2. **Budget Manager** - Handles financial planning and budgeting
3. **Market Researcher** - Provides market insights and recommendations

In [8]:
from crewai import Agent

# Agent 1: Script Analyst
script_analyst = Agent(
    role="Script Analyst",
    goal="Analyze movie scripts for sentiment, themes, and recommend appropriate soundtracks",
    backstory="""You are an experienced script analyst with 15 years in Hollywood.
    You have a keen eye for emotional depth and understand how music enhances storytelling.
    You've worked on numerous award-winning films and know what makes a script compelling.""",
    tools=[analyze_script_sentiment, recommend_soundtrack_genre],
    llm=llm,
    verbose=True,  # Shows detailed thinking process
    allow_delegation=False,  # Can't delegate tasks to other agents
)

# Agent 2: Budget Manager
budget_manager = Agent(
    role="Production Budget Manager",
    goal="Calculate accurate production budgets and optimize spending across departments",
    backstory="""You are a meticulous financial expert specializing in film production.
    With an MBA and 10 years of experience managing budgets for indie to blockbuster films,
    you ensure projects stay financially viable while maintaining creative vision.""",
    tools=[calculate_production_budget],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# Agent 3: Market Researcher
market_researcher = Agent(
    role="Film Market Research Analyst",
    goal="Provide data-driven insights about market trends, audience preferences, and optimal release strategies",
    backstory="""You are a data scientist specializing in entertainment industry analytics.
    You analyze box office trends, streaming data, and audience demographics to help studios
    make informed decisions about production and distribution.""",
    tools=[fetch_market_data],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Created 3 specialized agents:")
print("   1. Script Analyst (sentiment & soundtrack)")
print("   2. Budget Manager (financial planning)")
print("   3. Market Researcher (market intelligence)")

✅ Created 3 specialized agents:
   1. Script Analyst (sentiment & soundtrack)
   2. Budget Manager (financial planning)
   3. Market Researcher (market intelligence)


## Step 7: Define Tasks for Agents

Tasks are specific assignments given to agents. Each task has:
- A clear description
- An assigned agent
- Expected output format

In [9]:
from crewai import Task

# Sample movie concept for our tasks
movie_concept = """
Title: "Echoes of Tomorrow"
Genre: Sci-Fi Thriller
Logline: A brilliant AI researcher discovers a hidden message in quantum data that reveals
humanity's tragic future. Racing against time and a mysterious organization, she must decide
whether to alter the timeline or accept humanity's fate.

Script Excerpt: "In the depths of the unknown laboratory, Sarah's discovery brings both
triumph and terror. The hidden secret she uncovers could save millions, but the mystery
of who left this message haunts her every waking moment."
"""

# Task 1: Analyze the script
task_analyze_script = Task(
    description=f"""Analyze the following movie concept and script excerpt:

    {movie_concept}

    Use your tools to:
    1. Analyze the sentiment of the script excerpt
    2. Recommend appropriate soundtrack genres based on the genre and mood
    3. Provide creative suggestions for enhancing the emotional impact
    """,
    expected_output="A detailed analysis including sentiment breakdown, soundtrack recommendations, and creative suggestions",
    agent=script_analyst,
)

# Task 2: Calculate budget
task_budget_calculation = Task(
    description="""Based on the sci-fi thriller movie concept, calculate the production budget.

    Assume:
    - This is a MEDIUM budget production
    - 60 days of shooting
    - Crew size of 85 people

    Provide a detailed breakdown and recommendations for budget optimization.
    """,
    expected_output="Complete budget breakdown with optimization recommendations",
    agent=budget_manager,
)

# Task 3: Market research
task_market_research = Task(
    description="""Conduct market research for the sci-fi thriller genre.

    Use the MCP tool to fetch market data and provide:
    1. Current market performance of sci-fi films
    2. Target audience insights
    3. Optimal release timing
    4. Strategic recommendations based on trending themes
    """,
    expected_output="Comprehensive market analysis with strategic release recommendations",
    agent=market_researcher,
)

print("✅ Created 3 tasks:")
print("   1. Script Analysis Task")
print("   2. Budget Calculation Task")
print("   3. Market Research Task")

✅ Created 3 tasks:
   1. Script Analysis Task
   2. Budget Calculation Task
   3. Market Research Task


## Step 8: Assemble the Crew and Execute

Now we bring it all together! A Crew coordinates multiple agents working on related tasks.

**Process Types:**
- `sequential`: Tasks execute one after another (used here)
- `hierarchical`: A manager agent coordinates other agents

**Note:** Jupyter already runs an asyncio event loop, and CrewAI refuses to run synchronously inside one. In notebooks we therefore use `await crew.kickoff_async()`; in a regular Python script, use `crew.kickoff()`.

In [10]:
from crewai import Crew, Process

# Assemble the crew
movie_production_crew = Crew(
    agents=[script_analyst, budget_manager, market_researcher],
    tasks=[task_analyze_script, task_budget_calculation, task_market_research],
    process=Process.sequential,  # Tasks run in order
    verbose=True,  # Enable detailed output for learning
)

print("✅ Crew assembled with 3 agents and 3 tasks")
print("\n🚀 Starting production analysis...\n")
print("=" * 80)

# Execute the crew's tasks
result = await movie_production_crew.kickoff_async()

print("\n" + "=" * 80)
print("\n✅ Analysis Complete!\n")

✅ Crew assembled with 3 agents and 3 tasks

🚀 Starting production analysis...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0da3122a-8d17-4baa-8975-bd553a1ec657                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following movie concept and script excerpt:                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Title: "Echoes of Tomorrow"                                                                                    │
│  Genre: Sci-Fi Thriller                                                                                         │
│  Logline: A brilliant AI researcher discovers a hidden message in quantum data that reveals                     │
│  humanity's tragic future. Racing against time and a mysterious organization, she must decide                   │
│  whether to alter the timeline or accept humanity's fate.                                                       │
│                                                                                                                 │
│  Script Excerpt: "In the depths of the unknown laboratory, Sarah's discovery brings both                        │
│  triumph and terror. The hidden secret she uncovers could save millions, but the mystery                        │
│  of who left this message haunts her every waking moment."                                                      │
│                                                                                                                 │
│                                                                                                                 │
│      Use your tools to:                                                                                         │
│      1. Analyze the sentiment of the script excerpt                                                             │
│      2. Recommend appropriate soundtrack genres based on the genre and mood                                     │
│      3. Provide creative suggestions for enhancing the emotional impact                                         │
│                                                                                                                 │
│  ID: 787bac0a-4856-4c9c-97d6-cadd3845167e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Script Analyst                                                                                          │
│                                                                                                                 │
│  Task: Analyze the following movie concept and script excerpt:                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Title: "Echoes of Tomorrow"                                                                                    │
│  Genre: Sci-Fi Thriller                                                                                         │
│  Logline: A brilliant AI researcher discovers a hidden message in quantum data that reveals                     │
│  humanity's tragic future. Racing against time and a mysterious organization, she must decide                   │
│  whether to alter the timeline or accept humanity's fate.                                                       │
│                                                                                                                 │
│  Script Excerpt: "In the depths of the unknown laboratory, Sarah's discovery brings both                        │
│  triumph and terror. The hidden secret she uncovers could save millions, but the mystery                        │
│  of who left this message haunts her every waking moment."                                                      │
│                                                                                                                 │
│                                                                                                                 │
│      Use your tools to:                                                                                         │
│      1. Analyze the sentiment of the script excerpt                                                             │
│      2. Recommend appropriate soundtrack genres based on the genre and mood                                     │
│      3. Provide creative suggestions for enhancing the emotional impact                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Using an approximate input token count for model 'gpt-5.4-mini': assuming one token per four characters because a native token count is unavailable.


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: script_sentiment_analyzer                                                                                │
│  Args: {'script_text': "In the depths of the unknown laboratory, Sarah's discovery brings both triumph and      │
│  terror. The hidden secret she uncovers could save millions, but the mystery of who left this message...        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (19 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Conduct market research for the sci-fi thriller genre.                                                   │
│                                                                                                                 │
│      Use the MCP tool to fetch market data and provide:                                                         │
│      1. Current market performance of sci-fi films                                                              │
│      2. Target audience insights                                                                                │
│      3. Optimal release timing                                                                                  │
│      4. Strategic recommendations based on trending themes                                                      │
│                                                                                                                 │
│  Agent: Film Market Research Analyst                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0da3122a-8d17-4baa-8975-bd553a1ec657                                                                       │
│  Final Output: Below is a comprehensive market analysis for the **sci-fi thriller genre**, using the MCP        │
│  market data as the foundation and translating it into actionable release and positioning strategy for a        │
│  project like **“Echoes of Tomorrow.”**                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Sci-Fi Thriller Market Research Report                                                                       │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  Sci-fi remains one of the most commercially reliable premium genres when it pairs **high-concept               │
│  storytelling** with **clear emotional stakes** and **spectacle or novelty**. For a sci-fi thriller             │
│  specifically, the strongest market opportunities come from:                                                    │
│                                                                                                                 │
│  - **Event-style theatrical positioning**                                                                       │
│  - **Strong opening-weekend awareness**                                                                         │
│  - **Genre-aware audience targeting**                                                                           │
│  - **Concept-forward marketing built around mystery, technology, and stakes**                                   │
│                                                                                                                 │
│  Based on the market data:                                                                                      │
│  - **Average box office for sci-fi films:** **$380M**                                                           │
│  - **Primary audience:** **18–40 years**                                                                        │
│  - **Gender split:** **60% male**                                                                               │
│  - **Best release window:** **Summer and Holiday season**                                                       │
│  - **Trending themes:** **AI & technology, space exploration, time travel**                                     │
│  - **Streaming performance:** **High, with 88% completion rate**                                                │
│                                                                                                                 │
│  For a sci-fi thriller like *Echoes of Tomorrow*, the market favors a release strategy that emphasizes:         │
│  1. **theatrical event positioning**                                                                            │
│  2. **strong digital/streaming lifecycle planning**   



✅ Analysis Complete!



╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Step 9: View Final Results

Let's examine the consolidated output from all agents.

In [11]:
print("📋 FINAL PRODUCTION REPORT")
print("=" * 80)
print(result)
print("=" * 80)

📋 FINAL PRODUCTION REPORT
Below is a comprehensive market analysis for the **sci-fi thriller genre**, using the MCP market data as the foundation and translating it into actionable release and positioning strategy for a project like **“Echoes of Tomorrow.”**

---

# Sci-Fi Thriller Market Research Report

## Executive Summary

Sci-fi remains one of the most commercially reliable premium genres when it pairs **high-concept storytelling** with **clear emotional stakes** and **spectacle or novelty**. For a sci-fi thriller specifically, the strongest market opportunities come from:

- **Event-style theatrical positioning**
- **Strong opening-weekend awareness**
- **Genre-aware audience targeting**
- **Concept-forward marketing built around mystery, technology, and stakes**

Based on the market data:
- **Average box office for sci-fi films:** **$380M**
- **Primary audience:** **18–40 years**
- **Gender split:** **60% male**
- **Best release window:** **Summer and Holiday season**
- **Trendi

## Step 10: Implementing Observability

Observability helps you monitor and debug your agents. Let's add logging and tracking capabilities.

**Key Observability Features:**
1. Execution time tracking
2. Tool usage monitoring
3. Error logging
4. Agent decision tracking

In [12]:
import time
import json
from datetime import datetime

class CrewObserver:
    """Custom observability class for monitoring CrewAI execution"""

    def __init__(self):
        self.logs = []
        self.start_time = None
        self.end_time = None

    def log_event(self, event_type: str, agent_name: str, details: dict):
        """Log an event during crew execution"""
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'event_type': event_type,
            'agent': agent_name,
            'details': details
        }
        self.logs.append(log_entry)

    def start_monitoring(self):
        """Start monitoring crew execution"""
        self.start_time = time.time()
        self.log_event('CREW_START', 'System', {'message': 'Crew execution started'})

    def end_monitoring(self):
        """End monitoring and calculate metrics"""
        self.end_time = time.time()
        duration = self.end_time - self.start_time
        self.log_event('CREW_END', 'System', {
            'message': 'Crew execution completed',
            'duration_seconds': round(duration, 2)
        })

    def get_metrics(self):
        """Generate execution metrics"""
        if not self.end_time:
            return "Monitoring still in progress"

        duration = self.end_time - self.start_time
        agent_events = {}

        for log in self.logs:
            agent = log['agent']
            if agent != 'System':
                agent_events[agent] = agent_events.get(agent, 0) + 1

        metrics = f"""\n📊 EXECUTION METRICS\n{'='*60}
⏱️  Total Duration: {duration:.2f} seconds
📝 Total Events Logged: {len(self.logs)}
🤖 Agent Activity:"""

        for agent, count in agent_events.items():
            metrics += f"\n   • {agent}: {count} events"

        return metrics

    def display_logs(self, filter_agent=None):
        """Display execution logs"""
        print(f"\n📋 EXECUTION LOGS ({len(self.logs)} entries)")
        print("=" * 80)

        for log in self.logs:
            if filter_agent and log['agent'] != filter_agent:
                continue

            timestamp = log['timestamp'].split('T')[1].split('.')[0]
            print(f"[{timestamp}] {log['event_type']:15} | {log['agent']:25} | {log['details']}")

# Initialize observer
observer = CrewObserver()
print("✅ Observability system initialized")

✅ Observability system initialized


## Step 11: Run Crew with Observability

Now let's run our crew again with full observability enabled.

In [13]:
# Create a new task with observability hooks
observer.start_monitoring()

# Log agent initialization
observer.log_event('AGENT_INIT', 'Script Analyst', {'tools': ['sentiment_analyzer', 'soundtrack_recommender']})
observer.log_event('AGENT_INIT', 'Budget Manager', {'tools': ['budget_calculator']})
observer.log_event('AGENT_INIT', 'Market Researcher', {'tools': ['market_data_mcp']})

# New movie concept for second run
new_concept = """
Title: "The Last Symphony"
Genre: Drama
Logline: An elderly composer, losing her memory to dementia, races to complete her
final masterpiece while her estranged daughter learns to understand her through music.

Script Excerpt: "Each note brings both joy and sadness as Margaret's fingers dance across
the piano keys. The wonderful melody emerges from the fog of her fading memory, a
triumph over the terrible disease that threatens to steal her life's work."
"""

# Create new task
observed_task = Task(
    description=f"""Analyze this movie concept:
    {new_concept}

    Perform sentiment analysis and recommend soundtrack genres.""",
    expected_output="Sentiment analysis and soundtrack recommendations",
    agent=script_analyst,
)

# Log task start
observer.log_event('TASK_START', 'Script Analyst', {'task': 'Script analysis'})

# Create a smaller crew for demonstration
observed_crew = Crew(
    agents=[script_analyst],
    tasks=[observed_task],
    process=Process.sequential,
    verbose=False,  # Less verbose for cleaner output
)

print("🔍 Running crew with observability enabled...\n")

# Execute
result_observed = await observed_crew.kickoff_async()

# Log completion
observer.log_event('TASK_COMPLETE', 'Script Analyst', {'status': 'success'})
observer.end_monitoring()

print("\n✅ Execution complete with observability tracking!")

🔍 Running crew with observability enabled...



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Script Analyst                                                                                          │
│                                                                                                                 │
│  Task: Analyze this movie concept:                                                                              │
│                                                                                                                 │
│  Title: "The Last Symphony"                                                                                     │
│  Genre: Drama                                                                                                   │
│  Logline: An elderly composer, losing her memory to dementia, races to complete her                             │
│  final masterpiece while her estranged daughter learns to understand her through music.                         │
│                                                                                                                 │
│  Script Excerpt: "Each note brings both joy and sadness as Margaret's fingers dance across                      │
│  the piano keys. The wonderful melody emerges from the fog of her fading memory, a                              │
│  triumph over the terrible disease that threatens to steal her life's work."                                    │
│                                                                                                                 │
│                                                                                                                 │
│      Perform sentiment analysis and recommend soundtrack genres.                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Script Analyst                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Sentiment Analysis**                                                                                         │
│                                                                                                                 │
│  - **Overall tone:** Uplifting, bittersweet, and emotionally resonant                                           │
│  - **Positive sentiment:** 60%                                                                                  │
│  - **Negative sentiment:** 40%                                                                                  │
│  - **Dominant emotions:** Joy, sadness, hope, perseverance, tenderness                                          │
│                                                                                                                 │
│  **Analysis of the excerpt:**                                                                                   │
│  The passage carries a deeply emotional duality: Margaret’s playing is described as both joyful and sorrowful,  │
│  which creates a poignant sense of beauty amid decline. The image of melody “emerging from the fog of her       │
│  fading memory” adds vulnerability and fragility, while the phrase “a triumph over the terrible disease”        │
│  injects defiance and emotional victory. This is classic cathartic drama—heartache tempered by resilience.      │
│                                                                                                                 │
│  **Soundtrack Recommendations**                                                                                 │
│  For a **drama** with an **emotional** mood, the best-fitting soundtrack genres are:                            │
│                                                                                                                 │
│  1. **Piano Solo**                                                                                              │
│  2. **String Quartet**                                                                                          │
│  3. **Acoustic Guitar**                                                                                         │
│                                                                                                                 │
│  **Best musical direction for this concept:**                                                                   │
│  - **Piano solo** for Margaret’s intimacy, memory, and musical identity                                         │
│  - **String quartet** to underscore family tension, loss, and emotional depth                                   │
│  - **Acoustic guitar** for warmer, human moments between mother and daughter                                    │
│                                                                                                                 │
│  **Recommended blend:**                                                                                         │
│  A **piano-led score with string accompaniment** would be the strongest match, especially for scenes centered   │
│  on memory, performance, and reconciliation.                                                                    │
│                                                                                                                 │
╰────────────────────────────────────────────────────────


✅ Execution complete with observability tracking!


## 📈 Step 12: Analyze Observability Data

Let's examine the metrics and logs collected during execution.

In [14]:
# Display metrics
print(observer.get_metrics())

# Display all logs
observer.display_logs()

# Show result
print("\n📋 AGENT OUTPUT")
print("=" * 80)
print(result_observed)
print("=" * 80)


📊 EXECUTION METRICS
⏱️  Total Duration: 3.58 seconds
📝 Total Events Logged: 7
🤖 Agent Activity:
   • Script Analyst: 3 events
   • Budget Manager: 1 events
   • Market Researcher: 1 events

📋 EXECUTION LOGS (7 entries)
[16:44:55] CREW_START      | System                    | {'message': 'Crew execution started'}
[16:44:55] AGENT_INIT      | Script Analyst            | {'tools': ['sentiment_analyzer', 'soundtrack_recommender']}
[16:44:55] AGENT_INIT      | Budget Manager            | {'tools': ['budget_calculator']}
[16:44:55] AGENT_INIT      | Market Researcher         | {'tools': ['market_data_mcp']}
[16:44:55] TASK_START      | Script Analyst            | {'task': 'Script analysis'}
[16:44:58] TASK_COMPLETE   | Script Analyst            | {'status': 'success'}
[16:44:58] CREW_END        | System                    | {'message': 'Crew execution completed', 'duration_seconds': 3.58}

📋 AGENT OUTPUT
**Sentiment Analysis**

- **Overall tone:** Uplifting, bittersweet, and emotionally res

## Step 13: Advanced Observability - Export Logs

In production, you'd want to export logs for analysis. Here's how to save them.

In [15]:
import json

# Export logs to JSON
def export_logs(observer, filename='crew_execution_logs.json'):
    """Export observability logs to a JSON file"""
    log_data = {
        'execution_summary': {
            'start_time': datetime.fromtimestamp(observer.start_time).isoformat(),
            'end_time': datetime.fromtimestamp(observer.end_time).isoformat(),
            'duration_seconds': round(observer.end_time - observer.start_time, 2),
            'total_events': len(observer.logs)
        },
        'events': observer.logs
    }

    with open(filename, 'w') as f:
        json.dump(log_data, f, indent=2)

    return f"✅ Logs exported to {filename}"

# Export the logs
print(export_logs(observer))

# Display a sample of the JSON structure
print("\n📄 Sample JSON structure:")
sample = {
    'execution_summary': 'metadata about the run',
    'events': [
        {
            'timestamp': '2025-11-30T10:30:45.123456',
            'event_type': 'TASK_START',
            'agent': 'Script Analyst',
            'details': {'task': 'Script analysis'}
        }
    ]
}
print(json.dumps(sample, indent=2))

✅ Logs exported to crew_execution_logs.json

📄 Sample JSON structure:
{
  "execution_summary": "metadata about the run",
  "events": [
    {
      "timestamp": "2025-11-30T10:30:45.123456",
      "event_type": "TASK_START",
      "agent": "Script Analyst",
      "details": {
        "task": "Script analysis"
      }
    }
  ]
}


## 🎯 CHALLENGE: Build Your Own Movie Production Crew!

Now it's your turn to apply what you've learned! Complete the following challenge:

### 🎬 Challenge Description

Create a **Movie Casting Assistant** crew that helps with casting decisions for a new film.

### Requirements:

1. **Create TWO custom tools:**
   - `actor_availability_checker`: Takes an actor name and date range, returns if they're available (simulate with logic)
   - `role_matcher`: Takes character description and returns suggested actor traits (age range, experience level, acting style)

2. **Create ONE MCP-style tool:**
   - `fetch_actor_database`: Simulates querying a database of actors with their filmography, awards, and availability

3. **Create TWO agents:**
   - **Casting Director**: Uses role_matcher and fetch_actor_database
   - **Schedule Coordinator**: Uses actor_availability_checker

4. **Define tasks:**
   - Task 1: Find suitable actors for a lead role (give a character description)
   - Task 2: Check availability of suggested actors for specific shooting dates

5. **Add observability:**
   - Use the CrewObserver to track execution
   - Export logs at the end

### 💡 Bonus Points:
- Make your character descriptions creative and unique
- Add personality to your agents' backstories
- Include error handling in your tools
- Create a visualization of the observability data

### Sample Character (feel free to create your own!):
```
Character: Captain Zara Quinn
Description: A 35-year-old space captain with a mysterious past, known for her
quick wit and exceptional piloting skills. Must convey both strength and vulnerability.
```

### 📝 Your Code Goes Below:

Good luck! 🚀

In [16]:
import re
from datetime import datetime
from crewai.tools import tool

# Step 1: Create your custom tools
@tool("Actor Availability Checker")
def actor_availability_checker(actor_name: str, start_date: str, end_date: str) -> str:
    """
    Checks whether an actor is available for a shooting date range.

    Args:
        actor_name: Full name of the actor (e.g., 'Maya Chen')
        start_date: First shooting day in YYYY-MM-DD format
        end_date: Last shooting day in YYYY-MM-DD format
    """
    booked_periods = {
        'maya chen': [('2027-03-01', '2027-05-15', 'Starfall Protocol II')],
        'elena vasquez': [('2027-07-01', '2027-09-30', 'Crimson Tide Station')],
        'priya raman': [],
        'jordan blake': [('2027-05-01', '2027-06-15', 'Velocity Run')],
        'sofia lindqvist': [('2027-01-10', '2027-02-28', 'The Glass Fjord')],
        'daniel okafor': [('2027-04-01', '2027-12-31', 'Kingdom of Ash (TV series)')],
    }

    try:
        requested_start = datetime.strptime(start_date, '%Y-%m-%d')
        requested_end = datetime.strptime(end_date, '%Y-%m-%d')
    except ValueError:
        return f"Invalid date format ('{start_date}', '{end_date}'). Use YYYY-MM-DD."

    if requested_end < requested_start:
        return f"End date {end_date} is before start date {start_date}."

    actor_key = actor_name.strip().lower()
    if actor_key not in booked_periods:
        return f"No schedule found for '{actor_name}'. Known actors: {', '.join(name.title() for name in booked_periods.keys())}"

    for booked_start, booked_end, project in booked_periods[actor_key]:
        booked_start_date = datetime.strptime(booked_start, '%Y-%m-%d')
        booked_end_date = datetime.strptime(booked_end, '%Y-%m-%d')
        if requested_start <= booked_end_date and booked_start_date <= requested_end:
            return f"{actor_name} is NOT available from {start_date} to {end_date}: booked on '{project}' from {booked_start} to {booked_end}."

    return f"{actor_name} is available from {start_date} to {end_date}."

@tool("Role Matcher")
def role_matcher(character_description: str) -> str:
    """
    Analyzes a character description and suggests the actor traits needed for the role:
    age range, experience level and acting style.

    Args:
        character_description: A description of the character, including age and personality
    """
    description_lower = character_description.lower()

    age_match = re.search(r'(\d{1,2})[- ]year[- ]old', description_lower)
    if age_match:
        character_age = int(age_match.group(1))
        min_age = character_age - 5
        max_age = character_age + 5
        age_range = f"{min_age}-{max_age}"
    else:
        age_range = "Any (no age found in description)"

    lead_keywords = ['captain', 'lead', 'hero', 'mysterious past', 'strength and vulnerability', 'complex']
    lead_count = sum(1 for keyword in lead_keywords if keyword in description_lower)
    if lead_count >= 2:
        experience_level = "Seasoned (10+ years, has carried a lead role)"
    elif lead_count == 1:
        experience_level = "Experienced (5-10 years)"
    else:
        experience_level = "Emerging talent (0-5 years)"

    style_keywords = {
        'dramatic range': ['vulnerab', 'mysterious', 'tragic', 'emotional', 'past'],
        'comedic timing': ['wit', 'funny', 'humor', 'charming'],
        'physical/stunt': ['pilot', 'action', 'fight', 'soldier', 'athletic'],
    }
    acting_styles = []
    for style, keywords in style_keywords.items():
        if any(keyword in description_lower for keyword in keywords):
            acting_styles.append(style)
    if not acting_styles:
        acting_styles.append('naturalistic')

    result = "Role Analysis:\n"
    result += f"\nSuggested Age Range: {age_range}"
    result += f"\nExperience Level: {experience_level}"
    result += f"\nRequired Acting Styles: {', '.join(acting_styles)}"
    result += f"\n\nTip: Query the actor database with this age range and the style '{acting_styles[0]}'."

    return result


# Step 2: Create your MCP-style tool
@tool("Actor Database MCP")
def fetch_actor_database(min_age: int, max_age: int, acting_style: str = "any") -> str:
    """
    Queries the actor database for actors in an age range with a specific acting style.
    Returns their filmography, awards and availability.
    This simulates an MCP tool that connects to an external casting database.

    Args:
        min_age: Minimum actor age
        max_age: Maximum actor age
        acting_style: Required acting style (dramatic range, comedic timing, physical/stunt) or 'any'
    """
    actor_database = [
        {
            'name': 'Maya Chen',
            'age': 36,
            'experience_years': 14,
            'acting_styles': ['dramatic range', 'physical/stunt', 'comedic timing'],
            'filmography': ['Starfall Protocol (2024)', 'The Quiet Orbit (2021)', 'Neon Harbor (2019)'],
            'awards': ['Saturn Award - Best Actress (2024)'],
            'availability': 'Booked 2027-03-01 to 2027-05-15 (Starfall Protocol II)',
        },
        {
            'name': 'Elena Vasquez',
            'age': 33,
            'experience_years': 11,
            'acting_styles': ['dramatic range', 'physical/stunt'],
            'filmography': ['Dust Runner (2025)', 'Midnight Cartography (2022)', 'Iron Meridian (2020)'],
            'awards': ['Critics Circle - Breakthrough Performance (2022)'],
            'availability': 'Booked 2027-07-01 to 2027-09-30 (Crimson Tide Station)',
        },
        {
            'name': 'Priya Raman',
            'age': 38,
            'experience_years': 16,
            'acting_styles': ['dramatic range', 'comedic timing'],
            'filmography': ['The Lighthouse Keeper\'s Daughter (2023)', 'Monsoon Letters (2020)', 'Quantum Hearts (2018)'],
            'awards': ['Independent Spirit - Best Lead (2023)', 'Festival Jury Prize (2020)'],
            'availability': 'No current bookings',
        },
        {
            'name': 'Jordan Blake',
            'age': 29,
            'experience_years': 7,
            'acting_styles': ['physical/stunt', 'comedic timing'],
            'filmography': ['Velocity (2024)', 'Skyline Heist (2022)'],
            'awards': [],
            'availability': 'Booked 2027-05-01 to 2027-06-15 (Velocity Run)',
        },
        {
            'name': 'Sofia Lindqvist',
            'age': 42,
            'experience_years': 20,
            'acting_styles': ['dramatic range'],
            'filmography': ['Northern Silence (2024)', 'The Archivist (2019)', 'Winter Ledger (2015)'],
            'awards': ['European Film Award - Best Actress (2019)'],
            'availability': 'Booked 2027-01-10 to 2027-02-28 (The Glass Fjord)',
        },
        {
            'name': 'Daniel Okafor',
            'age': 51,
            'experience_years': 25,
            'acting_styles': ['dramatic range', 'comedic timing'],
            'filmography': ['Kingdom of Ash (2025)', 'The Ferryman (2017)'],
            'awards': ['Best Supporting Actor - National Film Awards (2017)'],
            'availability': 'Booked 2027-04-01 to 2027-12-31 (Kingdom of Ash TV series)',
        },
    ]

    if min_age > max_age:
        return f"Invalid age range: min_age ({min_age}) is greater than max_age ({max_age})."

    style_key = acting_style.strip().lower()

    matches = []
    for actor in actor_database:
        if actor['age'] < min_age or actor['age'] > max_age:
            continue
        if style_key != 'any' and style_key not in actor['acting_styles']:
            continue
        matches.append(actor)

    if not matches:
        return f"No actors found aged {min_age}-{max_age} with style '{acting_style}'."

    result = f"Actor Database Results (age {min_age}-{max_age}, style: {acting_style}):\n"
    for actor in matches:
        result += f"\n{actor['name']} (age {actor['age']}, {actor['experience_years']} years experience)"
        result += f"\n   Filmography: {', '.join(actor['filmography'])}"
        result += f"\n   Awards: {', '.join(actor['awards']) if actor['awards'] else 'None yet'}"
        result += f"\n   Acting Styles: {', '.join(actor['acting_styles'])}"
        result += f"\n   Availability: {actor['availability']}\n"
    result += "\nData retrieved from Actor Database (MCP)"

    return result


# Step 3: Create your agents
casting_director = Agent(
    role="Casting Director",
    goal="Find the best-suited actors for each role based on the character's traits and the actors' track records",
    backstory="""You are a legendary casting director who has discovered more breakout stars than anyone
    in the business. You trust your gut, but you always back it up with data from the actor database.
    You believe that great casting is 90% of directing, and you never settle for 'good enough'.""",
    tools=[role_matcher, fetch_actor_database],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)
schedule_coordinator = Agent(
    role="Schedule Coordinator",
    goal="Verify actor availability for the shooting dates and flag any scheduling conflicts",
    backstory="""You are an obsessively organized production coordinator who has never let a shoot
    slip a single day. You keep three calendars, double-check every date, and you can spot a
    scheduling conflict from across the studio lot.""",
    tools=[actor_availability_checker],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# Inputs for the casting tasks (the sample character from the challenge)
character_description = """Character: Captain Zara Quinn
Description: A 35-year-old space captain with a mysterious past, known for her
quick wit and exceptional piloting skills. Must convey both strength and vulnerability."""

shooting_start_date = "2027-04-15"
shooting_end_date = "2027-06-30"

# Step 4: Define your tasks
task_find_actors = Task(
    description=f"""Find suitable actors for the lead role in the sci-fi film "Echoes of Tomorrow".

    {character_description}

    Use your tools to:
    1. Analyze the character description with the Role Matcher to get the age range, experience level and acting styles
    2. Query the Actor Database with that age range and the most important acting style
    3. Shortlist the 3 best-matching actors and explain why each fits the role
    """,
    expected_output="A shortlist of 3 actors with their age, experience, filmography, awards and a short justification for each",
    agent=casting_director,
)

task_check_availability = Task(
    description=f"""Check the availability of every actor shortlisted by the Casting Director
    for the shooting dates {shooting_start_date} to {shooting_end_date}.

    Use the Actor Availability Checker for each shortlisted actor, then:
    1. List which actors are available and which have conflicts (with the conflicting project)
    2. Recommend the best available actor for the lead role
    """,
    expected_output="An availability report for each shortlisted actor and a final casting recommendation",
    agent=schedule_coordinator,
    context=[task_find_actors],
)

# Step 5: Set up observability
casting_observer = CrewObserver()
casting_observer.start_monitoring()

casting_observer.log_event('AGENT_INIT', 'Casting Director', {'tools': ['role_matcher', 'fetch_actor_database']})
casting_observer.log_event('AGENT_INIT', 'Schedule Coordinator', {'tools': ['actor_availability_checker']})

casting_crew = Crew(
    agents=[casting_director, schedule_coordinator],
    tasks=[task_find_actors, task_check_availability],
    process=Process.sequential,
    verbose=True,
)

casting_observer.log_event('TASK_START', 'Casting Director', {'task': 'Find suitable actors'})
casting_observer.log_event('TASK_START', 'Schedule Coordinator', {'task': 'Check actor availability'})

casting_result = await casting_crew.kickoff_async()

for task_output in casting_result.tasks_output:
    casting_observer.log_event('TASK_COMPLETE', task_output.agent, {'status': 'success', 'summary': task_output.summary})

casting_observer.end_monitoring()

# Step 6: Create and run your crew
# Hint: in a notebook, use `result = await crew.kickoff_async()`
print(casting_observer.get_metrics())

casting_observer.display_logs()

print("\nFINAL CASTING REPORT")
print(casting_result)

if casting_result.token_usage:
    print("\nToken Usage:")
    print(f"   Total tokens: {casting_result.token_usage.total_tokens}")
    print(f"   Prompt tokens: {casting_result.token_usage.prompt_tokens}")
    print(f"   Completion tokens: {casting_result.token_usage.completion_tokens}")
    print(f"   Successful requests: {casting_result.token_usage.successful_requests}")

print(export_logs(casting_observer, 'casting_crew_execution_logs.json'))

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4f51e70d-80ab-4a21-88a5-129895d965c8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find suitable actors for the lead role in the sci-fi film "Echoes of Tomorrow".                          │
│                                                                                                                 │
│      Character: Captain Zara Quinn                                                                              │
│  Description: A 35-year-old space captain with a mysterious past, known for her                                 │
│  quick wit and exceptional piloting skills. Must convey both strength and vulnerability.                        │
│                                                                                                                 │
│      Use your tools to:                                                                                         │
│      1. Analyze the character description with the Role Matcher to get the age range, experience level and      │
│  acting styles                                                                                                  │
│      2. Query the Actor Database with that age range and the most important acting style                        │
│      3. Shortlist the 3 best-matching actors and explain why each fits the role                                 │
│                                                                                                                 │
│  ID: 3e4d856d-6336-4806-84bf-b33e91ab0090                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Casting Director                                                                                        │
│                                                                                                                 │
│  Task: Find suitable actors for the lead role in the sci-fi film "Echoes of Tomorrow".                          │
│                                                                                                                 │
│      Character: Captain Zara Quinn                                                                              │
│  Description: A 35-year-old space captain with a mysterious past, known for her                                 │
│  quick wit and exceptional piloting skills. Must convey both strength and vulnerability.                        │
│                                                                                                                 │
│      Use your tools to:                                                                                         │
│      1. Analyze the character description with the Role Matcher to get the age range, experience level and      │
│  acting styles                                                                                                  │
│      2. Query the Actor Database with that age range and the most important acting style                        │
│      3. Shortlist the 3 best-matching actors and explain why each fits the role                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: role_matcher                                                                                             │
│  Args: {'character_description': 'Captain Zara Quinn: A 35-year-old space captain with a mysterious past,       │
│  known for her quick wit and exceptional piloting skills. Must convey both strength and vulnerability....       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: role_matcher                                                                                             │
│  Output: Role Analysis:                                                                                         │
│                                                                                                                 │
│  Suggested Age Range: 30-40                                                                                     │
│  Experience Level: Seasoned (10+ years, has carried a lead role)                                                │
│  Required Acting Styles: dramatic range, comedic timing, physical/stunt                                         │
│                                                                                                                 │
│  Tip: Query the actor database with this age range and the style 'dramatic range'.                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: actor_database_mcp                                                                                       │
│  Args: {'min_age': 30, 'max_age': 40, 'acting_style': 'dramatic range'}                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (14 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Check the availability of every actor shortlisted by the Casting Director                                │
│      for the shooting dates 2027-04-15 to 2027-06-30.                                                           │
│                                                                                                                 │
│      Use the Actor Availability Checker for each shortlisted actor, then:                                       │
│      1. List which actors are available and which have conflicts (with the conflicting project)                 │
│      2. Recommend the best available actor for the lead role                                                    │
│                                                                                                                 │
│  Agent: Schedule Coordinator                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


📊 EXECUTION METRICS
⏱️  Total Duration: 12.76 seconds
📝 Total Events Logged: 8
🤖 Agent Activity:
   • Casting Director: 3 events
   • Schedule Coordinator: 3 events

📋 EXECUTION LOGS (8 entries)
[16:44:58] CREW_START      | System                    | {'message': 'Crew execution started'}
[16:44:58] AGENT_INIT      | Casting Director          | {'tools': ['role_matcher', 'fetch_actor_database']}
[16:44:58] AGENT_INIT      | Schedule Coordinator      | {'tools': ['actor_availability_checker']}
[16:44:58] TASK_START      | Casting Director          | {'task': 'Find suitable actors'}
[16:44:58] TASK_START      | Schedule Coordinator      | {'task': 'Check actor availability'}
[16:45:11] TASK_COMPLETE   | Casting Director          | {'status': 'success', 'summary': 'Find suitable actors for the lead role in the sci-fi...'}
[16:45:11] TASK_COMPLETE   | Schedule Coordinator      | {'status': 'success', 'summary': 'Check the availability of every actor shortlisted by the Casting...'}
[16:45:

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4f51e70d-80ab-4a21-88a5-129895d965c8                                                                       │
│  Final Output: ## Availability Report: Captain Zara Quinn shortlist                                             │
│  **Shooting window:** 2027-04-15 to 2027-06-30                                                                  │
│                                                                                                                 │
│  ### 1) Maya Chen                                                                                               │
│  - **Availability:** **Not available**                                                                          │
│  - **Conflict:** Booked on **Starfall Protocol II** from **2027-03-01 to 2027-05-15**                           │
│  - **Impact on our shoot:** Her booking overlaps the first portion of the shooting window, so she cannot be     │
│  safely scheduled across the full required dates.                                                               │
│                                                                                                                 │
│  ### 2) Priya Raman                                                                                             │
│  - **Availability:** **Available**                                                                              │
│  - **Conflict:** None                                                                                           │
│  - **Status:** Clear for the full shooting window of **2027-04-15 to 2027-06-30**                               │
│                                                                                                                 │
│  ### 3) Elena Vasquez                                                                                           │
│  - **Availability:** **Available**                                                                              │
│  - **Conflict:** None                                                                                           │
│  - **Status:** Clear for the full shooting window of **2027-04-15 to 2027-06-30**                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Available actors vs. conflicts                                                                              │
│                                                                                                                 │
│  ### Available                                                                                                  │
│  - **Priya Raman**                                                                                              │
│  - **Elena Vasquez**                                                                                            │
│                                                                                                                 │
│  ### Have conflicts                                                                                             │
│  - **Maya Chen** — conflict with **Starfall Protocol II** (2027-03-01 to 2027-05-15)                            │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 🎉 Congratulations!

You've completed the CrewAI practice notebook! You've learned:

✅ How to create custom tools for agents  
✅ How to implement MCP-style tools for external integrations  
✅ How to configure and use an Azure OpenAI model  
✅ How to create specialized agents with specific roles  
✅ How to define and orchestrate tasks  
✅ How to implement observability and monitoring  
✅ How to export and analyze execution logs  

### 🔗 Useful Resources:

- [CrewAI Documentation](https://docs.crewai.com/)
- [CrewAI LLM Configuration](https://docs.crewai.com/en/concepts/llms)
- [CrewAI Tools](https://docs.crewai.com/en/concepts/tools)
- [Azure OpenAI v1 API](https://learn.microsoft.com/azure/ai-foundry/openai/api-version-lifecycle)

Happy coding! 🚀🎬